# 03 卷积层运算规则

上一节我们从整体上认识了 CNN：它会先提取图像特征，再完成分类。

这一节开始学习 CNN 里最核心的部分：卷积层。

这节仍然先不写代码，也不讲 PyTorch API。

这一节只解决一个问题：**卷积层到底是怎么在图片上做运算的。**

## 1. 先把卷积层想成一个小窗口

卷积层可以先理解成一个在图片上滑动的小窗口。

这个小窗口里面有一些参数，这些参数就是卷积核里的权重。

小窗口每次盖住图片的一小块区域，然后和这块区域做一次计算，得到一个数字。

这个小窗口继续往右、往下滑动，就会得到很多数字。

这些数字重新排成一张新的二维表，就叫特征图。

可以先这样记：

```text
卷积核在图片上滑动，每个位置算出一个值，所有值组成特征图。
```

## 2. 什么是卷积核

卷积核也叫 filter，可以先理解成一组用来识别局部特征的权重。

比如一个 3 x 3 的卷积核，就是一个 3 行 3 列的小矩阵：

$$
\begin{bmatrix}
w_1 & w_2 & w_3 \\
w_4 & w_5 & w_6 \\
w_7 & w_8 & w_9
\end{bmatrix}
$$

它一次只看图片中的 3 x 3 小区域。

不同卷积核可以学习不同特征。

有的卷积核可能更容易对横线有反应，有的可能更容易对竖线有反应，有的可能更容易对斜线或边缘有反应。

不过这些卷积核的具体数值通常不是我们手写出来的，而是模型在训练中自己学出来的。

## 3. 卷积的一次计算怎么做

先看最简单的单通道灰度图。

假设图片中的一个 3 x 3 小区域是：

$$
\begin{bmatrix}
1 & 2 & 0 \\
0 & 1 & 3 \\
2 & 1 & 1
\end{bmatrix}
$$

卷积核是：

$$
\begin{bmatrix}
1 & 0 & -1 \\
1 & 0 & -1 \\
1 & 0 & -1
\end{bmatrix}
$$

计算规则是：对应位置相乘，然后全部加起来。

$$
1\times1 + 2\times0 + 0\times(-1)
+ 0\times1 + 1\times0 + 3\times(-1)
+ 2\times1 + 1\times0 + 1\times(-1)
$$

算出来就是：

$$
1 + 0 + 0 + 0 + 0 - 3 + 2 + 0 - 1 = -1
$$

所以这个位置的输出值就是 -1。

一句话总结：

```text
卷积的一次计算 = 局部区域和卷积核对应位置相乘，再求和。
```

## 4. 为什么会得到一张特征图

刚才只算了一个位置。

但是卷积核不会只看一个地方。

它会从图片左上角开始，依次往右滑动，再往下滑动。

每滑到一个位置，就做一次对应位置相乘再求和。

所以：

```text
一个位置 -> 一个输出数字
很多位置 -> 很多输出数字
很多输出数字排起来 -> 一张特征图
```

特征图可以理解成：这个卷积核在图片各个位置找到某种特征的强弱程度。

## 5. 步长 stride 是什么

卷积核滑动时，每次移动几格，这个移动距离就叫步长，英文是 stride。

如果 stride = 1，表示每次移动 1 个像素。

如果 stride = 2，表示每次移动 2 个像素。

步长越大，卷积核跳得越快，输出的特征图通常就越小。

可以这样理解：

```text
stride 小：看得更细，输出更大。
stride 大：跳得更快，输出更小。
```

入门阶段先常见 stride = 1。

## 6. padding 是什么

padding 叫填充。

它的意思是在图片四周补一圈或多圈数值，通常补 0。

为什么要补？

因为如果不补，卷积核在图片边缘位置不好完整覆盖，输出尺寸会变小。

比如 28 x 28 的图片，用 3 x 3 卷积核，stride = 1，如果不加 padding，输出会变成 26 x 26。

如果四周补一圈 0，也就是 padding = 1，输出就可以保持 28 x 28。

可以先这样记：

```text
padding 的作用之一：控制输出特征图的大小，避免边缘信息太快丢失。
```

## 7. 输出尺寸怎么计算

卷积层的输出高宽可以用一个公式计算。

先看高度方向：

$$
H_{out}=\left\lfloor\frac{H_{in}+2P-K}{S}\right\rfloor+1
$$

宽度方向同理：

$$
W_{out}=\left\lfloor\frac{W_{in}+2P-K}{S}\right\rfloor+1
$$

其中：

- $H_{in}$：输入图片或特征图的高度。
- $W_{in}$：输入图片或特征图的宽度。
- $K$：卷积核大小。这里先假设卷积核是 K x K。
- $P$：padding，填充圈数。
- $S$：stride，步长。
- $H_{out}$、$W_{out}$：输出特征图的高度和宽度。

这个公式不需要死背，重点是理解每个量在影响什么。

## 8. 输出尺寸例子一：不加 padding

假设输入是 MNIST 图片：

$$
H_{in}=28,\quad W_{in}=28
$$

卷积核大小是 3 x 3：

$$
K=3
$$

不加 padding：

$$
P=0
$$

步长是 1：

$$
S=1
$$

代入公式：

$$
H_{out}=\frac{28+2\times0-3}{1}+1=26
$$

宽度同样是 26。

所以输出特征图大小是：

$$
26\times26
$$

## 9. 输出尺寸例子二：加 padding

还是输入 28 x 28 的 MNIST 图片。

卷积核大小仍然是 3 x 3，步长仍然是 1。

这次加一圈 padding：

$$
P=1
$$

代入公式：

$$
H_{out}=\frac{28+2\times1-3}{1}+1=28
$$

宽度同样是 28。

所以输出特征图大小还是：

$$
28\times28
$$

这就是为什么 3 x 3 卷积核经常配 padding = 1：它可以在 stride = 1 时保持高宽不变。

## 10. 单通道和多通道卷积

前面的例子都是灰度图，只有 1 个通道。

如果是 RGB 彩色图，就有 3 个通道。

这时卷积核也不能只是一张二维小表。

它要同时覆盖输入的所有通道。

比如输入是 RGB 图片，卷积核大小是 3 x 3，那么一个卷积核实际上有 3 层：

```text
R 通道对应一个 3 x 3 小矩阵
G 通道对应一个 3 x 3 小矩阵
B 通道对应一个 3 x 3 小矩阵
```

计算时，每个通道分别做乘加，最后把所有通道的结果加在一起，得到一个输出数字。

所以多通道卷积可以先记成：

```text
每个输入通道都参与计算，最后合成一个输出值。
```

## 11. 一个卷积核得到几个输出通道

一个卷积核在整张图片上滑动，会得到一张特征图。

这一张特征图就是 1 个输出通道。

所以：

```text
1 个卷积核 -> 1 张特征图 -> 1 个输出通道
```

如果有 16 个卷积核，就会得到 16 张特征图，也就是 16 个输出通道。

这也是为什么 CNN 里经常会看到通道数从 1 变成 16，再从 16 变成 32。

通道数变多，不是图片变多了，而是模型提取出来的特征种类变多了。

## 12. 卷积层的参数量怎么理解

卷积层的参数来自卷积核。

如果输入有 $C_{in}$ 个通道，卷积核大小是 K x K，一个卷积核需要的权重数量是：

$$
C_{in}\times K\times K
$$

如果有 $C_{out}$ 个卷积核，也就是要输出 $C_{out}$ 个通道，那么总权重数量是：

$$
C_{out}\times C_{in}\times K\times K
$$

如果每个输出通道还有一个偏置，偏置数量就是：

$$
C_{out}
$$

入门阶段先不必急着背参数量公式。

先理解一个关键点：卷积核的参数会在整张图上重复使用，这叫权值共享。

## 13. 权值共享是什么意思

权值共享是 CNN 很重要的特点。

同一个卷积核在图片不同位置滑动时，用的是同一组权重。

这意味着：

```text
同一种特征检测方法，可以在整张图片上反复使用。
```

比如一个卷积核学会检测斜线。

这条斜线出现在左上角也好，出现在右下角也好，这个卷积核都可以去检测它。

这就是 CNN 比普通全连接网络更适合图像的一个重要原因。

## 14. 本节小结

这一节先记住这些规则：

1. 卷积核可以理解成在图片上滑动的小窗口。
2. 一次卷积计算，就是局部区域和卷积核对应位置相乘再求和。
3. 一个卷积核滑完整张图，会得到一张特征图。
4. stride 表示卷积核每次移动几格。
5. padding 表示在图片周围补几圈数，常用于控制输出大小。
6. 一个卷积核对应一个输出通道，多个卷积核对应多个输出通道。
7. 多通道输入时，卷积核也要覆盖所有输入通道。
8. 权值共享表示同一个卷积核在不同位置重复使用同一组参数。

## 自检问题

1. 卷积核在图片上滑动时，每个位置会输出几个数字？
2. 卷积的一次计算规则是什么？
3. stride 变大时，输出特征图通常会变大还是变小？
4. padding 的作用是什么？
5. 28 x 28 的图片，用 3 x 3 卷积核、stride = 1、padding = 0，输出大小是多少？
6. 28 x 28 的图片，用 3 x 3 卷积核、stride = 1、padding = 1，输出大小是多少？
7. 一个卷积核会产生几个输出通道？
8. 如果有 16 个卷积核，会产生几个输出通道？
9. 多通道卷积为什么不是只看其中一个通道？
10. 什么是权值共享？